# 12 Super Data Visualization

ภาพ interactive ชุดนี้เล่าเรื่องทั้งโปรเจกต์ด้วยข้อมูลจริง เปิดใน notebook หรือเปิดหน้ารวม `outputs/super_viz.html` ในเบราว์เซอร์ก็ได้

1. A day on BART: แผนที่เคลื่อนไหวทีละชั่วโมง คนเข้าและออกแต่ละสถานี ปี 2019 เทียบกับ 2023
2. เส้นทางการเดินทาง: 150 คู่สถานีที่มีคนมากที่สุด และทริปที่หายไป
3. 12 ปีในภาพเดียว: ทุกชั่วโมงของทุกวันตั้งแต่ปี 2011
4. ตัวสำรวจผลพยากรณ์: ค่าจริงเทียบกับพยากรณ์ล่วงหน้า 7 วันของ M1 จากการทดสอบย้อนหลัง (M3)
5. ภาพเสริม: การฟื้นตัวรายสถานีแบบเคลื่อนไหว, สถานีที่เปลี่ยนประเภท (M2), Transbay Tube 13 ปี

พิกัดสถานีและเส้นรางมาจาก GTFS ทางการของ BART (ดู `data/reference/SOURCE.md`) แผนที่พื้นหลังเป็น tile ของ CARTO / OpenStreetMap จึงต้องต่อเน็ตตอนเปิดดู
ภาพที่เคลื่อนไหวได้ ไฟล์ PNG จะเก็บแค่สถานะเริ่มต้น ถ้าจะดูแบบเคลื่อนไหวให้เปิดไฟล์ HTML

In [1]:
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import math
from datetime import date, timedelta

import numpy as np
import plotly.graph_objects as go
import polars as pl

from bart import viz
from bart.config import FIG, MODELS, OUT, TAB, create_views, duck
from bart.perf import stage
from bart.periods import AFTER, BEFORE, period_case
from bart.stations import station_frame

viz.notebook_mode()

REFERENCE = ROOT / "data" / "reference"
con = duck()
create_views(con)

geo = pl.read_csv(REFERENCE / "stations_geo.csv").select("station", "lat", "lon")
stations = station_frame().join(geo, on="station", how="left")
missing = stations.filter(pl.col("lat").is_null())["station"].to_list()
if len(missing) > 0:
    raise ValueError(f"stations without coordinates: {missing}")

lines = pl.read_csv(REFERENCE / "bart_lines.csv").sort("line", "seq")
print(f"stations with coordinates: {stations.height}, track points: {lines.height}")

stations with coordinates: 50, track points: 1630


### ส่วนที่ใช้ร่วมกัน

- เส้นรางจริงของแต่ละสาย วาดเป็นพื้นหลังสีจางๆ ตามสีสายของ BART
- ฟังก์ชันตั้งค่าแผนที่ ให้ทุกแผนที่ใช้มุมมองเดียวกัน

In [2]:
MAP_CENTER = dict(lat=37.70, lon=-122.13)
LINE_ORDER = ["Yellow", "Orange", "Green", "Red", "Blue", "Grey"]


def track_traces(subplot):
    traces = []
    for line_name in LINE_ORDER:
        part = lines.filter(pl.col("line") == line_name)
        traces.append(go.Scattermap(
            lat=part["lat"], lon=part["lon"], mode="lines",
            line=dict(color=part["color"][0], width=2), opacity=0.45,
            name=f"{line_name} line", showlegend=False, hoverinfo="skip", subplot=subplot,
        ))
    return traces


def map_layout(zoom, x_domain):
    return dict(
        style="carto-positron", center=MAP_CENTER, zoom=zoom,
        domain=dict(x=x_domain, y=[0, 1]),
    )

## 1. A day on BART

ค่าเฉลี่ยคนเข้า (entries) และคนออก (exits) ของแต่ละสถานี ในแต่ละชั่วโมงของวันทำงานปกติ ม.ค.-เม.ย. 2019 เทียบกับ ม.ค.-เม.ย. 2023
ขนาดวงกลมใช้สเกลเดียวกันทั้งสองปี กด Play เพื่อดูทั้งวัน หรือเลื่อน slider ไปชั่วโมงที่ต้องการ

In [3]:
with stage("12.day_profile"):
    profile = con.sql(f"""
        WITH tagged AS (
            SELECT {period_case("s.date")} AS period, s.station, s.hour, s.entries, s.exits
            FROM station_hour s
            JOIN calendar c ON s.date = c.date
            WHERE c.day_status = 'normal' AND c.is_workday
        )
        SELECT period, station, hour, avg(entries) AS entries, avg(exits) AS exits
        FROM tagged
        WHERE period IS NOT NULL
        GROUP BY period, station, hour
    """).pl().join(stations, on="station")

SHOW_HOURS = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 0]
START_HOUR = 8
BUBBLE_MAX_PX = 46

largest = max(float(profile.select(pl.col("entries").max()).item()), float(profile.select(pl.col("exits").max()).item()))
SIZEREF = 2.0 * largest / (BUBBLE_MAX_PX ** 2)

hour_totals = profile.group_by("period", "hour").agg(pl.col("entries").sum())


def hour_total(period, hour):
    return float(hour_totals.filter((pl.col("period") == period) & (pl.col("hour") == hour))["entries"][0])


print(f"largest station-hour average: {largest:,.0f}")
for hour in [8, 12, 17]:
    before = hour_total(BEFORE, hour)
    after = hour_total(AFTER, hour)
    print(f"{hour:02d}:00 entries, whole system: {before:,.0f} -> {after:,.0f} ({100 * after / before:.0f}% of 2019)")

[perf] 12.day_profile: 0.02s | 3.92 cores | peak RAM 0.199 GB
largest station-hour average: 12,483
08:00 entries, whole system: 50,760 -> 16,908 (33% of 2019)
12:00 entries, whole system: 11,069 -> 5,357 (48% of 2019)
17:00 entries, whole system: 49,434 -> 17,934 (36% of 2019)


In [ ]:
PANELS = [(BEFORE, "map"), (AFTER, "map2")]
FLOWS = [("exits", "exits (arriving)", viz.CONTRAST), ("entries", "entries (leaving from)", viz.ACCENT)]


def bubble_traces(hour):
    traces = []
    for period, subplot in PANELS:
        part = profile.filter((pl.col("period") == period) & (pl.col("hour") == hour))
        for column, label, color in FLOWS:
            traces.append(go.Scattermap(
                lat=part["lat"], lon=part["lon"], mode="markers",
                marker=dict(size=part[column], sizemode="area", sizeref=SIZEREF, sizemin=2,
                            color=color, opacity=0.55),
                name=label, legendgroup=column, showlegend=(subplot == "map"),
                customdata=np.column_stack([part["station"].to_numpy(), part["name"].to_numpy()]),
                hovertemplate="%{customdata[1]} (%{customdata[0]})<br>" + column + ": %{marker.size:,.0f} per hour<extra>" + period + "</extra>",
                subplot=subplot,
            ))
    return traces


def hour_annotations(hour):
    notes = []
    x_positions = [0.0, 0.51]
    for position, (period, subplot) in enumerate(PANELS):
        share = 100 * hour_total(period, hour) / hour_total(BEFORE, hour)
        text = f"<b>{period}</b>  {hour:02d}:00"
        if period == AFTER:
            text += f"  ({share:.0f}% of 2019 entries)"
        notes.append(dict(text=text, x=x_positions[position], y=1.0, xref="paper", yref="paper",
                        xanchor="left", yanchor="bottom", showarrow=False, font=dict(size=14, color=viz.INK)))
    return notes


fig = go.Figure()
for period, subplot in PANELS:
    for trace in track_traces(subplot):
        fig.add_trace(trace)
first_bubble = len(LINE_ORDER) * len(PANELS)
for trace in bubble_traces(START_HOUR):
    fig.add_trace(trace)
bubble_indices = list(range(first_bubble, first_bubble + len(PANELS) * len(FLOWS)))

frames = []
slider_steps = []
for hour in SHOW_HOURS:
    name = f"{hour:02d}:00"
    frames.append(go.Frame(name=name, data=bubble_traces(hour), traces=bubble_indices,
                           layout=dict(annotations=hour_annotations(hour))))
    slider_steps.append(dict(label=name, method="animate",
                             args=[[name], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))]))
fig.frames = frames

play = dict(label="Play", method="animate",
            args=[None, dict(frame=dict(duration=800, redraw=True), fromcurrent=True, transition=dict(duration=0))])
pause = dict(label="Pause", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate", transition=dict(duration=0))])

fig.update_layout(
    map=map_layout(8.5, [0.0, 0.49]),
    map2=map_layout(8.5, [0.51, 1.0]),
    annotations=hour_annotations(START_HOUR),
    updatemenus=[dict(type="buttons", buttons=[play, pause], direction="left", x=0, y=-0.04,
                      xanchor="left", yanchor="top", showactive=False)],
    sliders=[dict(steps=slider_steps, active=SHOW_HOURS.index(START_HOUR), x=0.12, len=0.88, y=-0.02,
                  currentvalue=dict(visible=False), pad=dict(t=0))],
    legend=dict(y=1.09),
    margin=dict(l=20, r=20, t=150, b=90),
    hovermode="closest",
)
morning_share = 100 * hour_total(AFTER, 8) / hour_total(BEFORE, 8)
title = f"A workday on BART: the 8:00 rush is now {morning_share:.0f}% of its 2019 size"
subtitle = "Average entries and exits per station and hour, workdays Jan-Apr; press Play to run through the day"
viz.titled(fig, title, subtitle)
viz.save(fig, "12_day_on_bart", width=1400, height=760)
day_figure = fig
fig

ตอน 8:00 ปี 2019 วงกลมสีส้มก้อนใหญ่ที่สุดอยู่ที่ downtown SF คือคนจากทุกสายมาลงที่เดียวกัน ปี 2023 ก้อนนี้ยังอยู่ที่เดิม แต่เล็กลงมาก คนเข้าทั้งระบบตอน 8:00 เหลือ 33% ของปี 2019 ส่วนตอน 12:00 เหลือ 48% และ 17:00 เหลือ 36%
- รูปแบบการไหลยังเหมือนเดิม คือเช้าเข้าเมืองและเย็นออกจากเมือง (กด Play แล้วดูว่าสีส้มย้ายจาก SF ไปชานเมืองช่วงเย็น) ที่เปลี่ยนคือขนาด สอดคล้องกับ Q2 ที่ peak เตี้ยลงแต่ไม่ได้หายไป
- MLPT และ BERY ไม่มีวงกลมในปี 2019 เพราะเปิดเดือน มิ.ย. 2020

## 2. เส้นทางการเดินทาง

เส้นตรงเชื่อมต้นทางกับปลายทาง (desire line) ของ 150 คู่สถานีที่มีคนมากที่สุดในวันทำงาน ม.ค.-เม.ย. 2019 นับรวมทั้งสองทิศ เส้นไม่ได้วิ่งตามราง แต่บอกว่าคนต้องการไปจากไหนถึงไหน
- สีส้ม = ทริปที่ข้าม Transbay Tube สีน้ำเงิน = ทริปที่อยู่ฝั่งเดียวของอ่าว
- ความหนาใช้สเกลเดียวกันทุกมุมมอง กดปุ่มด้านบนเพื่อสลับระหว่าง 2019, 2023 และทริปที่หายไป

In [5]:
with stage("12.od_pairs"):
    od_pairs = con.sql(f"""
        WITH tagged AS (
            SELECT {period_case("month")} AS period, least(origin, dest) AS station_a,
                   greatest(origin, dest) AS station_b, trips
            FROM od_month
            WHERE day_type = 'workday' AND origin <> dest
        ),
        pair_trips AS (
            SELECT period, station_a, station_b, sum(trips) AS trips
            FROM tagged
            WHERE period IS NOT NULL
            GROUP BY period, station_a, station_b
        ),
        period_days AS (
            SELECT period, sum(days) AS days
            FROM (SELECT {period_case("month")} AS period, days FROM month_days WHERE day_type = 'workday')
            WHERE period IS NOT NULL
            GROUP BY period
        )
        SELECT p.period, p.station_a, p.station_b, p.trips / d.days AS per_day
        FROM pair_trips p
        JOIN period_days d ON p.period = d.period
    """).pl()

wide = od_pairs.pivot(on="period", index=["station_a", "station_b"], values="per_day").fill_null(0)
wide = wide.rename({BEFORE: "before", AFTER: "after"}).with_columns(
    (pl.col("before") - pl.col("after")).alias("lost"),
)

TOP_PAIRS = 150
top_pairs = wide.sort("before", descending=True).head(TOP_PAIRS)

ends = stations.select("station", "lat", "lon", "side")
top_pairs = (
    top_pairs
    .join(ends.rename({"station": "station_a", "lat": "lat_a", "lon": "lon_a", "side": "side_a"}), on="station_a")
    .join(ends.rename({"station": "station_b", "lat": "lat_b", "lon": "lon_b", "side": "side_b"}), on="station_b")
    .with_columns((pl.col("side_a") != pl.col("side_b")).alias("crosses_bay"))
)

share_top = 100 * float(top_pairs["before"].sum()) / float(wide["before"].sum())
crossing_before = float(top_pairs.filter(pl.col("crosses_bay"))["before"].sum())
crossing_after = float(top_pairs.filter(pl.col("crosses_bay"))["after"].sum())
local_before = float(top_pairs.filter(~pl.col("crosses_bay"))["before"].sum())
local_after = float(top_pairs.filter(~pl.col("crosses_bay"))["after"].sum())
print(f"top {TOP_PAIRS} pairs carry {share_top:.1f}% of 2019 workday trips")
print(f"crossing the bay : {crossing_before:,.0f} -> {crossing_after:,.0f} per workday ({100 * crossing_after / crossing_before:.0f}%)")
print(f"same side of bay : {local_before:,.0f} -> {local_after:,.0f} per workday ({100 * local_after / local_before:.0f}%)")
print(f"pairs crossing the bay: {top_pairs['crosses_bay'].sum()} of {TOP_PAIRS}")

[perf] 12.od_pairs: 0.01s | 4.11 cores | peak RAM 0.29 GB
top 150 pairs carry 64.9% of 2019 workday trips
crossing the bay : 168,745 -> 53,593 per workday (32%)
same side of bay : 96,049 -> 31,680 per workday (33%)
pairs crossing the bay: 100 of 150


In [6]:
WIDTH_BANDS = [(3000, 10.0, "3,000+"), (2000, 6.5, "2,000-3,000"), (1000, 3.5, "1,000-2,000"),
               (500, 1.8, "500-1,000"), (0, 0.8, "under 500")]
VIEWS = [("before", "Jan-Apr 2019"), ("after", "Jan-Apr 2023"), ("lost", "lost since 2019")]
SIDES = [(True, "crosses the bay", viz.CONTRAST), (False, "same side", viz.ACCENT)]


def band_of(value):
    for floor, width, label in WIDTH_BANDS:
        if value >= floor:
            return floor
    return 0


def segment_coordinates(rows):
    lat_values = []
    lon_values = []
    for row in rows.to_dicts():
        lat_values.extend([row["lat_a"], row["lat_b"], None])
        lon_values.extend([row["lon_a"], row["lon_b"], None])
    return lat_values, lon_values


fig = go.Figure()
for trace in track_traces("map"):
    fig.add_trace(trace)
static_count = len(LINE_ORDER)

view_of_trace = []
for column, view_label in VIEWS:
    banded = top_pairs.with_columns(
        pl.col(column).map_elements(band_of, return_dtype=pl.Int64).alias("band"),
    )
    for crosses, side_label, color in SIDES:
        for floor, width, band_label in WIDTH_BANDS:
            rows = banded.filter((pl.col("crosses_bay") == crosses) & (pl.col("band") == floor))
            lat_values, lon_values = segment_coordinates(rows)
            if len(lat_values) == 0:
                lat_values = [None]
                lon_values = [None]
            show_in_legend = (column == "before") and (floor == 1000)
            fig.add_trace(go.Scattermap(
                lat=lat_values, lon=lon_values, mode="lines",
                line=dict(color=color, width=width), opacity=0.7,
                name=side_label, legendgroup=side_label, showlegend=show_in_legend,
                hoverinfo="skip", visible=(column == "before"), subplot="map",
            ))
            view_of_trace.append(column)

station_hover = []
for row in stations.to_dicts():
    station_hover.append(f"{row['name']} ({row['station']})")
fig.add_trace(go.Scattermap(
    lat=stations["lat"], lon=stations["lon"], mode="markers",
    marker=dict(size=6, color=viz.INK), name="station", showlegend=False,
    text=station_hover, hovertemplate="%{text}<extra></extra>", subplot="map",
))

midpoint_lat = []
midpoint_lon = []
midpoint_text = []
for row in top_pairs.to_dicts():
    midpoint_lat.append((row["lat_a"] + row["lat_b"]) / 2)
    midpoint_lon.append((row["lon_a"] + row["lon_b"]) / 2)
    midpoint_text.append(
        f"{row['station_a']} - {row['station_b']}<br>2019: {row['before']:,.0f} per workday"
        f"<br>2023: {row['after']:,.0f} ({100 * row['after'] / row['before']:.0f}%)"
    )
fig.add_trace(go.Scattermap(
    lat=midpoint_lat, lon=midpoint_lon, mode="markers", marker=dict(size=8, color="rgba(0,0,0,0)"),
    text=midpoint_text, hovertemplate="%{text}<extra></extra>", showlegend=False, name="pair", subplot="map",
))
tail_count = 2

buttons = []
for column, view_label in VIEWS:
    visible = []
    for index in range(static_count):
        visible.append(True)
    for trace_view in view_of_trace:
        visible.append(trace_view == column)
    for index in range(tail_count):
        visible.append(True)
    buttons.append(dict(label=view_label, method="update", args=[dict(visible=visible)]))

fig.update_layout(
    map=map_layout(8.9, [0.0, 1.0]),
    updatemenus=[dict(type="buttons", buttons=buttons, direction="right", x=0, y=1.06,
                      xanchor="left", yanchor="bottom", active=0)],
    legend=dict(x=1, xanchor="right", y=1.06, yanchor="bottom"),
    margin=dict(l=20, r=20, t=170, b=70),
    hovermode="closest",
)
crossing_share = 100 * crossing_before / (crossing_before + local_before)
title = (f"{crossing_share:.0f}% of the busiest trips crossed the bay; crossing and local pairs both kept only "
         f"{100 * crossing_after / crossing_before:.0f}-{100 * local_after / local_before:.0f}%")
subtitle = (f"Top {TOP_PAIRS} station pairs of 2019 ({share_top:.0f}% of workday trips), line width = trips per workday, "
            "both directions; hover the middle of a line for numbers")
viz.titled(fig, title, subtitle)
band_labels = []
for floor, width, label in WIDTH_BANDS:
    band_labels.append(label)
viz.note(fig, "Width bands (trips per workday): " + ", ".join(band_labels) + ". Straight desire lines, not track paths.",
         y=-0.06)
viz.save(fig, "12_od_flows", width=1300, height=900)
od_figure = fig
fig

ปี 2019 เส้นสีส้มเกือบทั้งหมดพุ่งเข้า downtown SF จากทุกทิศ 100 จาก 150 คู่ข้ามอ่าว และคิดเป็น 64% ของทริปในกลุ่มนี้ เส้นข้ามอ่าวที่หนาที่สุดคือ DUBL, WOAK และ PHIL กับ EMBR (4,000-4,500 ทริปต่อวันทำงาน)
- กด Jan-Apr 2023 เส้นบางลงทั้งแผนที่ ทั้งคู่ที่ข้ามอ่าวและไม่ข้ามเหลือ 32-33% ใกล้กันมาก แปลว่าการหายไปไม่ได้เจาะจงที่ Tube แต่เกิดกับทุกเส้นทางที่เข้าเมือง
- กด lost since 2019 เส้นที่หนาที่สุดคือเส้นที่หายไปมากที่สุด เป็นการเดินทางไกลจากชานเมืองเข้า downtown ตรงกับ Q3

## 3. 12 ปีในภาพเดียว

ทุกช่องคือ 1 ชั่วโมงของ 1 วัน (4,505 วัน × 24 ชั่วโมง) สีคือจำนวนทริปทั้งระบบที่เริ่มในชั่วโมงนั้น วันที่ไม่มีข้อมูลเป็นช่องว่าง
ใช้ station แบบเดียวกับ `system_hour` ใน marts ซูมเข้าไปดูช่วงที่สนใจได้

In [7]:
with stage("12.twelve_years"):
    hours_grid = con.sql("""
        SELECT s.date, s.hour, s.trips
        FROM system_hour s
        ORDER BY s.date, s.hour
    """).pl()

grid = hours_grid.pivot(on="date", index="hour", values="trips").sort("hour")
date_columns = grid.columns[1:]
z_values = grid.select(date_columns).to_numpy().astype("float64")
day_axis = []
for name in date_columns:
    day_axis.append(date.fromisoformat(str(name)))

color_top = float(np.nanpercentile(z_values, 99.5))

fig = go.Figure(go.Heatmap(
    x=day_axis, y=grid["hour"].to_list(), z=z_values,
    colorscale=viz.SEQUENTIAL, zmin=0, zmax=color_top,
    colorbar=dict(title=dict(text="trips<br>per hour"), thickness=12, tickformat="~s"),
    hovertemplate="%{x|%a %d %b %Y} %{y}:00<br>%{z:,.0f} trips<extra></extra>",
))

MARKS = [
    (date(2013, 7, 1), "strikes Jul and Oct 2013"),
    (date(2020, 3, 17), "shelter in place 17 Mar 2020"),
]
for day, label in MARKS:
    fig.add_annotation(x=day, y=-0.5, yref="y", text=label, showarrow=True, arrowhead=0, ax=0, ay=-34,
                       arrowcolor=viz.INK2, font=dict(size=11, color=viz.INK2), bgcolor="rgba(255,255,255,0.8)")

fig.update_yaxes(title_text="hour of day", autorange="reversed", dtick=3, showgrid=False)
fig.update_xaxes(title_text=None, showgrid=False)
fig.update_layout(hovermode="closest", margin=dict(t=150))

year_2019 = float(hours_grid.filter(pl.col("date").dt.year() == 2019)["trips"].sum())
year_2022 = float(hours_grid.filter(pl.col("date").dt.year() == 2022)["trips"].sum())
title = f"12 years of BART, every hour: the two commute bands fade after March 2020 ({100 * year_2022 / year_2019:.0f}% of 2019 trips in 2022)"
subtitle = "System trips per hour, 1 Jan 2011 - 2 May 2023; each column is one day, each row one hour"
viz.titled(fig, title, subtitle)
viz.save(fig, "12_twelve_years", width=1500, height=620)
years_figure = fig
fig

[perf] 12.twelve_years: 0.01s | 0.0 cores | peak RAM 0.315 GB


ภาพนี้ใช้ข้อมูลทั้ง 12 ปีในรูปเดียว แถบสีเข้มสองแถบคือ peak เช้า (7-9 โมง) และเย็น (16-18 นาฬิกา) ที่เห็นต่อเนื่องตั้งแต่ปี 2011 จนถึงต้นปี 2020 แล้วจางลงทันทีหลัง shelter in place
- เส้นสีขาวในแนวตั้งคือวันที่ไม่มีข้อมูลและวัน strike ปี 2013 ส่วนลายขีดถี่ๆ คือวันเสาร์-อาทิตย์ที่ peak หายไป
- หลังปี 2020 แถบเช้าและเย็นค่อยๆ กลับมา แต่ทั้งปี 2022 มีทริปแค่ 35% ของปี 2019 และแถบกลางวันกับแถบเช้าใกล้กันขึ้น (peak แบนลง ตาม Q2)

## 4. ตัวสำรวจผลพยากรณ์

ใช้ผลจาก notebook 10 ทุกค่าเป็นการพยากรณ์ล่วงหน้า 7 วันจากโมเดลที่ไม่เคยเห็นช่วงนั้น (ทดสอบย้อนหลังทีละไตรมาส)

### 4.1 รายวัน ทุกสถานี
เลือกสถานีจาก dropdown เส้นสีเข้มคือคนเข้าจริงต่อวัน เส้นสีน้ำเงินคือพยากรณ์ P50 รวมทั้งวัน จุดสีส้มคือวันคนล้น จุดสีแดงคือวันคนหาย ตามเกณฑ์ของ M3 (เกินช่วง P10-P90 ที่ปรับแล้วรวมกันอย่างน้อย 25% ของ P50 และ 300 คน)

In [8]:
station_days = pl.read_parquet(MODELS / "m3_station_days.parquet").sort("station", "date")
calibrated = pl.read_parquet(MODELS / "m3_calibrated.parquet")

surge_by_station = station_days.group_by("station").agg(pl.col("surge").sum().alias("surge_days")).sort(
    "surge_days", descending=True)
explorer_stations = station_days["station"].unique().sort().to_list()
DEFAULT_STATION = surge_by_station["station"][0]
print(f"stations: {len(explorer_stations)}, days: {station_days['date'].n_unique()}, default: {DEFAULT_STATION}")

names = {}
for row in stations.to_dicts():
    names[row["station"]] = row["name"]

fig = go.Figure()
traces_per_station = 4
for station in explorer_stations:
    part = station_days.filter(pl.col("station") == station)
    surges = part.filter(pl.col("surge"))
    drops = part.filter(pl.col("drop"))
    visible = station == DEFAULT_STATION
    fig.add_trace(go.Scatter(
        x=part["date"], y=part["actual"], mode="lines", name="actual entries",
        line=dict(color=viz.INK, width=1.5), visible=visible,
        hovertemplate="%{x|%a %d %b %Y}: actual %{y:,.0f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=part["date"], y=part["p50"], mode="lines", name="forecast P50 (7 days ahead)",
        line=dict(color=viz.ACCENT, width=1.5), visible=visible,
        hovertemplate="%{x|%a %d %b %Y}: P50 %{y:,.0f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=surges["date"], y=surges["actual"], mode="markers", name="surge day",
        marker=dict(color=viz.CONTRAST, size=9, line=dict(color="white", width=1.5)), visible=visible,
        hovertemplate="%{x|%a %d %b %Y}: surge, actual %{y:,.0f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=drops["date"], y=drops["actual"], mode="markers", name="drop day",
        marker=dict(color=viz.SERIES[7], size=9, symbol="diamond", line=dict(color="white", width=1.5)),
        visible=visible,
        hovertemplate="%{x|%a %d %b %Y}: drop, actual %{y:,.0f}<extra></extra>",
    ))

buttons = []
for position, station in enumerate(explorer_stations):
    visible = []
    for other in explorer_stations:
        for index in range(traces_per_station):
            visible.append(other == station)
    surge_days = int(station_days.filter((pl.col("station") == station) & pl.col("surge")).height)
    drop_days = int(station_days.filter((pl.col("station") == station) & pl.col("drop")).height)
    label = f"{station} - {names[station]}"
    buttons.append(dict(label=label, method="update", args=[
        dict(visible=visible),
        dict(title=dict(text=f"<b>{names[station]} ({station}): actual vs forecast made 7 days ahead</b>"
                             f"<br><span style='font-size:13px;color:{viz.MUTED}'>{surge_days} surge days and "
                             f"{drop_days} drop days, Jul 2021 - May 2023</span>")),
    ]))

default_position = explorer_stations.index(DEFAULT_STATION)
fig.update_layout(
    updatemenus=[dict(buttons=buttons, active=default_position, x=1, xanchor="right", y=1.17, yanchor="bottom")],
    hovermode="x unified", margin=dict(t=160),
)
fig.update_yaxes(title_text="entries per day", tickformat="~s", rangemode="tozero")

default_surges = int(station_days.filter((pl.col("station") == DEFAULT_STATION) & pl.col("surge")).height)
default_drops = int(station_days.filter((pl.col("station") == DEFAULT_STATION) & pl.col("drop")).height)
viz.titled(fig, f"{names[DEFAULT_STATION]} ({DEFAULT_STATION}): actual vs forecast made 7 days ahead",
           f"{default_surges} surge days and {default_drops} drop days, Jul 2021 - May 2023")
viz.save(fig, "12_forecast_explorer", width=1300, height=620)
explorer_figure = fig
fig

stations: 50, days: 671, default: COLS


ค่าเริ่มต้นเป็น COLS เพราะมีวันคนล้นมากที่สุด 104 วัน (ใน notebook 10 นับ 103 วัน เพราะตัดวันพาเหรด 20 มิ.ย. 2022 ซึ่งเป็นเหตุการณ์ระดับเมืองออก) วันที่สูงที่สุดคือเสาร์ 21 ส.ค. 2021 คนเข้า 6,578 คน ขณะที่พยากรณ์ไว้ 887 คน
- ช่วงที่ไม่มีงาน เส้นจริงกับเส้นพยากรณ์ทับกันเกือบสนิท ลองเลือกสถานีอื่นจะเห็นจุดสีส้มน้อยกว่ามาก เช่น MONT 14 วัน และ DUBL 11 วัน
- จุดสีแดงของ MLBR ช่วง ธ.ค. 2021 คือวันที่คนเข้าเกือบเป็นศูนย์ที่พบใน notebook 10

### 4.2 รายชั่วโมง สัปดาห์ที่น่าสนใจ

ช่วง P10-P90 ที่ปรับแล้ว (แถบสีฟ้า) ใช้ได้กับค่ารายชั่วโมงเท่านั้น จึงดูแบบรายชั่วโมงทีละสัปดาห์ เลือกสัปดาห์จาก dropdown
สัปดาห์ COLS เลือกจากวันที่คนเกินช่วงมากที่สุดของสถานี ส่วนสัปดาห์อื่นเป็นวันที่พบใน notebook 10

In [9]:
cols_peak = station_days.filter(pl.col("station") == "COLS").sort("over", descending=True).row(0, named=True)
cols_monday = cols_peak["date"] - timedelta(days=cols_peak["date"].weekday())

WEEKS = [
    ("CIVC", date(2022, 6, 20), "CIVC, week of the Warriors parade (Mon) and Pride (Sun)"),
    ("EMBR", date(2022, 6, 20), "EMBR, week of the Warriors parade"),
    ("COLS", cols_monday, f"COLS, week of its biggest surge ({cols_peak['date']:%d %b %Y})"),
    ("MONT", date(2023, 1, 16), "MONT, January 2023 after holiday and storm weeks"),
    ("POWL", date(2023, 1, 2), "POWL, storm week of 4 Jan 2023"),
    ("MLBR", date(2021, 12, 20), "MLBR, near-zero days in December 2021"),
]

fig = go.Figure()
traces_per_week = 4
for position, (station, monday, label) in enumerate(WEEKS):
    week = calibrated.filter(
        (pl.col("station") == station) & pl.col("date").is_between(monday, monday + timedelta(days=6))
    ).sort("date", "hour").with_columns(
        (pl.col("date").cast(pl.Datetime) + pl.duration(hours=pl.col("hour"))).alias("time"),
    )
    visible = position == 0
    fig.add_trace(go.Scatter(x=week["time"], y=week["p90_cal"], mode="lines", line=dict(width=0),
                             showlegend=False, hoverinfo="skip", visible=visible))
    fig.add_trace(go.Scatter(x=week["time"], y=week["p10_cal"], mode="lines", line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(42, 120, 214, 0.18)", name="P10-P90 (calibrated)",
                             hoverinfo="skip", visible=visible))
    fig.add_trace(go.Scatter(x=week["time"], y=week["p50"], mode="lines", name="P50",
                             line=dict(color=viz.ACCENT, width=2), visible=visible,
                             hovertemplate="%{x|%a %d %b %H:00}: P50 %{y:,.0f}<extra></extra>"))
    fig.add_trace(go.Scatter(x=week["time"], y=week["target"], mode="lines", name="actual",
                             line=dict(color=viz.INK, width=2, dash="dot"), visible=visible,
                             hovertemplate="%{x|%a %d %b %H:00}: actual %{y:,.0f}<extra></extra>"))

week_buttons = []
for position, (station, monday, label) in enumerate(WEEKS):
    visible = []
    for other in range(len(WEEKS)):
        for index in range(traces_per_week):
            visible.append(other == position)
    week_buttons.append(dict(label=label, method="update", args=[
        dict(visible=visible),
        dict(title=dict(text=f"<b>{label}</b><br><span style='font-size:13px;color:{viz.MUTED}'>"
                             "Hourly entries vs forecast made 7 days ahead, with the calibrated P10-P90 band</span>")),
    ]))

fig.update_layout(
    updatemenus=[dict(buttons=week_buttons, active=0, x=1, xanchor="right", y=1.17, yanchor="bottom")],
    hovermode="x unified", margin=dict(t=160),
)
fig.update_yaxes(title_text="entries per hour", tickformat="~s", rangemode="tozero")
viz.titled(fig, WEEKS[0][2], "Hourly entries vs forecast made 7 days ahead, with the calibrated P10-P90 band")
viz.save(fig, "12_event_weeks", width=1300, height=560)
weeks_figure = fig
fig

สัปดาห์ 20-26 มิ.ย. 2022 ที่ CIVC เห็นเหตุการณ์สองครั้งในสัปดาห์เดียว
- จันทร์ 20 มิ.ย. (พาเหรด Warriors): 15:00 มีคนเข้า 2,434 คน ขณะที่ P90 ที่ปรับแล้วอยู่ที่ 625 คน
- อาทิตย์ 26 มิ.ย. (Pride): 17:00 มีคนเข้า 3,521 คน P90 อยู่ที่ 946 คน ทั้งวันมีคนเข้า 20,708 คน เกือบ 5 เท่าของพยากรณ์ (4,224 คน)

วันอื่นในสัปดาห์เดียวกัน ค่าจริงอยู่ในแถบเกือบตลอด แปลว่าโมเดลแม่นในวันปกติ และพลาดเฉพาะวันที่มีงานซึ่งโมเดลไม่มีข้อมูล ตรงกับข้อเสนอให้ทำปฏิทินงานใหญ่ของเมือง (สาเหตุของทั้งสองวันมาจากความรู้ภายนอก)

## 5. ภาพเสริม

### 5.1 การฟื้นตัวรายสถานี เดือนต่อเดือน
ค่าเฉลี่ยคนเข้าต่อวันทำงานปกติของแต่ละเดือน เทียบกับเดือนเดียวกันของปี 2019 ตั้งแต่ ม.ค. 2020 ถึง เม.ย. 2023 เรียงจากฟื้นมากไปน้อยในแต่ละเดือน สีตามประเภทสถานีปี 2019 จาก M2 (ไม่รวม MLPT และ BERY ที่ยังไม่มีในปี 2019)

In [10]:
with stage("12.recovery_race"):
    monthly = con.sql("""
        SELECT d.station, date_trunc('month', d.date) AS month, avg(d.entries) AS entries
        FROM station_day d
        JOIN calendar c ON d.date = c.date
        WHERE c.day_status = 'normal' AND c.is_workday AND d.date >= DATE '2019-01-01' AND d.date < DATE '2023-05-01'
        GROUP BY d.station, date_trunc('month', d.date)
    """).pl()

base_2019 = monthly.filter(pl.col("month").dt.year() == 2019).select(
    "station", pl.col("month").dt.month().alias("month_number"), pl.col("entries").alias("entries_2019"),
)
recovery = (
    monthly.filter(pl.col("month") >= date(2020, 1, 1))
    .with_columns(pl.col("month").dt.month().alias("month_number"))
    .join(base_2019, on=["station", "month_number"])
    .with_columns((100 * pl.col("entries") / pl.col("entries_2019")).alias("recovery"))
)

types = pl.read_csv(TAB / "m2_station_clusters.csv").filter(pl.col("period") == BEFORE).select("station", "cluster")
recovery = recovery.join(types, on="station", how="left")

TYPE_COLORS = {
    "commuter origin": viz.SERIES[0],
    "mixed origin": viz.SERIES[2],
    "downtown job centre": viz.SERIES[1],
    "all-day destination": viz.SERIES[3],
}
months = recovery["month"].unique().sort().to_list()
print(f"stations: {recovery['station'].n_unique()}, months: {len(months)} ({months[0]:%b %Y} - {months[-1]:%b %Y})")

def race_bars(month):
    part = recovery.filter(pl.col("month") == month).sort("recovery", descending=True)
    colors = []
    for cluster in part["cluster"].to_list():
        colors.append(TYPE_COLORS[cluster])
    ranks = list(range(1, part.height + 1))
    return go.Bar(
        x=part["recovery"], y=ranks, orientation="h", marker=dict(color=colors),
        text=part["station"], textposition="outside", cliponaxis=False,
        customdata=np.column_stack([part["station"].to_numpy(), part["cluster"].to_numpy()]),
        hovertemplate="%{customdata[0]} (%{customdata[1]}): %{x:.0f}% of 2019<extra></extra>",
        showlegend=False,
    )


START_MONTH = months[-1]
fig = go.Figure(race_bars(START_MONTH))
for cluster, color in TYPE_COLORS.items():
    fig.add_trace(go.Bar(x=[None], y=[None], marker=dict(color=color), name=cluster, showlegend=True))

race_frames = []
race_steps = []
for month in months:
    name = f"{month:%b %Y}"
    race_frames.append(go.Frame(name=name, data=[race_bars(month)], traces=[0],
                                layout=dict(annotations=[dict(text=name, x=0.98, y=0.04, xref="paper", yref="paper",
                                                            xanchor="right", showarrow=False,
                                                            font=dict(size=34, color=viz.CONTEXT))])))
    race_steps.append(dict(label=name, method="animate",
                        args=[[name], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))]))
fig.frames = race_frames

play = dict(label="Play", method="animate",
            args=[None, dict(frame=dict(duration=450, redraw=True), fromcurrent=True, transition=dict(duration=200))])
pause = dict(label="Pause", method="animate",
            args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate", transition=dict(duration=0))])
fig.update_layout(
    updatemenus=[dict(type="buttons", buttons=[play, pause], direction="left", x=0, y=-0.075,
                    xanchor="left", yanchor="top", showactive=False)],
    sliders=[dict(steps=race_steps, active=len(months) - 1, x=0.12, len=0.88, y=-0.06,
                currentvalue=dict(visible=False), pad=dict(t=0))],
    annotations=[dict(text=f"{START_MONTH:%b %Y}", x=0.98, y=0.04, xref="paper", yref="paper", xanchor="right",
                    showarrow=False, font=dict(size=34, color=viz.CONTEXT))],
    hovermode="closest", bargap=0.25, margin=dict(t=150, b=190),
)
fig.update_xaxes(title_text="workday entries, % of the same month in 2019", ticksuffix="%", range=[0, 130],
                showgrid=True, gridcolor=viz.GRID)
fig.update_yaxes(autorange="reversed", showticklabels=False, showgrid=False, title_text=None)

last = recovery.filter(pl.col("month") == START_MONTH).sort("recovery", descending=True)
title = f"By {START_MONTH:%B %Y} the best station is at {last['recovery'][0]:.0f}% of 2019, the weakest at {last['recovery'][-1]:.0f}%"
subtitle = "Station recovery month by month since January 2020; press Play to watch the fall and the slow climb"
viz.titled(fig, title, subtitle)
viz.save(fig, "12_recovery_race", width=1100, height=1200)
race_figure = fig
fig

[perf] 12.recovery_race: 0.01s | 2.85 cores | peak RAM 0.435 GB
stations: 48, months: 40 (Jan 2020 - Apr 2023)


เดือนท้าย (เม.ย. 2023) สถานีที่ฟื้นมากที่สุดคือ SFIA 62% และน้อยที่สุดคือ MONT 27% สีส้ม (ย่านทำงาน) กองอยู่ด้านล่าง ส่วนสีเหลือง (ปลายทางทั้งวัน เช่น สนามบิน) อยู่ด้านบน ตรงกับ M2 และ Q1
- กด Play จะเห็นทุกสถานีตกลงพร้อมกันใน เม.ย. 2020 แล้วค่อยๆ ไต่ขึ้น
- WARM อยู่ท้ายตาราง สอดคล้องกับ Q4 ที่ผู้โดยสารย้ายไป MLPT/BERY

### 5.2 สถานีที่เปลี่ยนประเภท (M2)

ความกว้างของแถบคือจำนวนสถานีที่ย้ายจากประเภทหนึ่งในปี 2019 ไปอีกประเภทในปี 2023

In [11]:
clusters = pl.read_csv(TAB / "m2_station_clusters.csv")
before_types = clusters.filter(pl.col("period") == BEFORE).select("station", pl.col("cluster").alias("type_2019"))
after_types = clusters.filter(pl.col("period") == AFTER).select("station", pl.col("cluster").alias("type_2023"))
moves = before_types.join(after_types, on="station").group_by("type_2019", "type_2023").agg(
    pl.len().alias("stations"), pl.col("station").sort().str.join(", ").alias("names"),
)

type_order = list(TYPE_COLORS.keys())
node_labels = []
node_colors = []
for cluster in type_order:
    node_labels.append(f"{cluster} 2019")
    node_colors.append(TYPE_COLORS[cluster])
for cluster in type_order:
    node_labels.append(f"{cluster} 2023")
    node_colors.append(TYPE_COLORS[cluster])

sources = []
targets = []
values = []
link_colors = []
link_text = []
for row in moves.to_dicts():
    sources.append(type_order.index(row["type_2019"]))
    targets.append(len(type_order) + type_order.index(row["type_2023"]))
    values.append(row["stations"])
    if row["type_2019"] == row["type_2023"]:
        link_colors.append("rgba(196, 195, 189, 0.45)")
    else:
        link_colors.append("rgba(235, 104, 52, 0.45)")
    link_text.append(row["names"])

fig = go.Figure(go.Sankey(
    arrangement="fixed",
    node=dict(label=node_labels, color=node_colors, pad=24, thickness=18, line=dict(width=0)),
    link=dict(source=sources, target=targets, value=values, color=link_colors, customdata=link_text,
              hovertemplate="%{source.label} to %{target.label}: %{value} stations<br>%{customdata}<extra></extra>"),
))
switched = int(moves.filter(pl.col("type_2019") != pl.col("type_2023"))["stations"].sum())
total = int(moves["stations"].sum())
viz.titled(fig, f"{switched} of {total} stations changed role between 2019 and 2023",
           "Station types from M2 (KMeans on hourly profiles); orange links = stations that changed type")
fig.update_layout(margin=dict(t=120, l=20, r=20, b=20), hovermode="closest")
viz.save(fig, "12_station_roles", width=1100, height=560)
roles_figure = fig
fig

25 จาก 48 สถานีเปลี่ยนประเภท แถบสีส้มกว้างที่สุดคือ ต้นทางคนทำงาน ไปเป็น ต้นทางแบบผสม (22 สถานี) ส่วนที่เหลืออีก 3 สถานีคือ WARM (ต้นทางคนทำงาน ไปเป็นย่านทำงาน), 16TH (ย่านทำงาน ไปเป็นปลายทางทั้งวัน) และ 19TH (ย่านทำงาน ไปเป็นต้นทางแบบผสม) ชี้เมาส์ที่แถบเพื่อดูรายชื่อสถานี

### 5.3 Transbay Tube 13 ปี

ทริปที่ข้ามอ่าวในแต่ละชั่วโมงของวันทำงานปกติ ม.ค.-เม.ย. ของทุกปี แยกทิศ เส้นจางคือปี 2019 ไว้เทียบ กด Play เพื่อดูทีละปี
อ่านจาก fact table (112 ล้านแถว) โดยตรง เพราะต้องรู้ต้นทางและปลายทางของทุกทริป

In [12]:
with stage("12.transbay_years"):
    tube = con.sql("""
        WITH sides AS (
            SELECT station, side FROM stations
        ),
        days AS (
            SELECT date, year FROM calendar
            WHERE day_status = 'normal' AND is_workday AND month(date) <= 4
        ),
        crossing AS (
            SELECT d.year, o.hour,
                    CASE WHEN so.side = 'east' THEN 'East Bay to SF / Peninsula' ELSE 'SF / Peninsula to East Bay' END AS direction,
                    sum(o.trips) AS trips
            FROM od o
            JOIN days d ON o.date = d.date
            JOIN sides so ON o.origin = so.station
            JOIN sides sd ON o.dest = sd.station
            WHERE so.side <> sd.side
            GROUP BY d.year, o.hour, so.side
        ),
        day_counts AS (
            SELECT year, count(*) AS days FROM days GROUP BY year
        )
        SELECT c.year, c.hour, c.direction, c.trips / n.days AS per_day
        FROM crossing c
        JOIN day_counts n ON c.year = n.year
        ORDER BY c.year, c.direction, c.hour
    """).pl()

years = tube["year"].unique().sort().to_list()
DIRECTIONS = [("East Bay to SF / Peninsula", viz.CONTRAST), ("SF / Peninsula to East Bay", viz.ACCENT)]
TUBE_HOURS = list(range(4, 24))


def tube_lines(year, ghost):
    traces = []
    for direction, color in DIRECTIONS:
        part = tube.filter((pl.col("year") == year) & (pl.col("direction") == direction) & pl.col("hour").is_in(TUBE_HOURS)).sort("hour")
        if ghost:
            traces.append(go.Scatter(x=part["hour"], y=part["per_day"], mode="lines", name=f"{direction}, 2019",
                                    line=dict(color=color, width=1.5, dash="dot"), opacity=0.5,
                                    hovertemplate="%{x}:00 2019: %{y:,.0f}<extra></extra>"))
        else:
            traces.append(go.Scatter(x=part["hour"], y=part["per_day"], mode="lines+markers", name=direction,
                                    line=dict(color=color, width=2.5), marker=dict(size=6),
                                    hovertemplate="%{x}:00: %{y:,.0f} trips per workday<extra>" + direction + "</extra>"))
    return traces


def tube_total(year):
    return float(tube.filter(pl.col("year") == year)["per_day"].sum())


fig = go.Figure()
for trace in tube_lines(2019, True):
    fig.add_trace(trace)
LAST_YEAR = years[-1]
for trace in tube_lines(LAST_YEAR, False):
    fig.add_trace(trace)


def year_note(year):
    share = 100 * tube_total(year) / tube_total(2019)
    return [dict(text=f"{year}", x=0.99, y=0.97, xref="paper", yref="paper", xanchor="right", yanchor="top",
                showarrow=False, font=dict(size=40, color=viz.CONTEXT)),
            dict(text=f"{tube_total(year):,.0f} crossings per workday ({share:.0f}% of 2019)", x=0.99, y=0.80,
                xref="paper", yref="paper", xanchor="right", yanchor="top", showarrow=False,
                font=dict(size=13, color=viz.INK2))]


tube_frames = []
tube_steps = []
for year in years:
    name = str(year)
    tube_frames.append(go.Frame(name=name, data=tube_lines(year, False), traces=[2, 3],
                                layout=dict(annotations=year_note(year))))
    tube_steps.append(dict(label=name, method="animate",
                        args=[[name], dict(mode="immediate", frame=dict(duration=0, redraw=False), transition=dict(duration=300))]))
fig.frames = tube_frames

play = dict(label="Play", method="animate",
            args=[None, dict(frame=dict(duration=900, redraw=False), fromcurrent=True, transition=dict(duration=400))])
pause = dict(label="Pause", method="animate",
            args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate", transition=dict(duration=0))])
peak = float(tube.select(pl.col("per_day").max()).item())
fig.update_layout(
    annotations=year_note(LAST_YEAR),
    updatemenus=[dict(type="buttons", buttons=[play, pause], direction="left", x=0, y=-0.2,
                    xanchor="left", yanchor="top", showactive=False)],
    sliders=[dict(steps=tube_steps, active=len(years) - 1, x=0.12, len=0.88, y=-0.17,
                currentvalue=dict(visible=False), pad=dict(t=0))],
    margin=dict(t=150, b=170), hovermode="x unified",
)
fig.update_xaxes(title_text="hour the trip started", dtick=2, range=[3.5, 23.5])
fig.update_yaxes(title_text="trips crossing the bay per workday", tickformat="~s", range=[0, peak * 1.08])

growth = 100 * tube_total(2019) / tube_total(years[0]) - 100
title = f"The Transbay Tube: {growth:.0f}% busier from {years[0]} to 2019, then {100 * tube_total(LAST_YEAR) / tube_total(2019):.0f}% of 2019 in {LAST_YEAR}"
subtitle = "Trips crossing the bay per workday by hour and direction, Jan-Apr of each year; dotted = 2019"
viz.titled(fig, title, subtitle)
viz.save(fig, "12_transbay_years", width=1200, height=640)
tube_figure = fig
fig

[perf] 12.transbay_years: 0.20s | 16.69 cores | peak RAM 0.552 GB | CPU 85.5%


ทริปข้ามอ่าวช่วง ม.ค.-เม.ย. เพิ่มจาก ~169,000 ต่อวันทำงานในปี 2011 สูงสุด ~235,000 ในปี 2016 แล้วทรงตัวถึงปี 2019 (~228,000 สูงกว่าปี 2011 35%) กด Play จะเห็นปี 2020 ลดเหลือ 57% ของปี 2019 (ม.ค.-ต้น มี.ค. ยังปกติ แต่ปลาย มี.ค.-เม.ย. แทบไม่มีคน) และปี 2021 เหลือแค่ 11%
- ปี 2023 ข้ามอ่าว 78,602 ทริปต่อวันทำงาน (34% ของปี 2019) peak ยังอยู่ที่ 8:00 ทิศไป SF และ 17:00 ทิศกลับ แต่เส้นเย็นของปี 2023 สูงสุดที่ 17:00 ไม่ใช่ 18:00 แบบปี 2019 ตรงกับ Q3

## 6. หน้ารวม

รวมทุกภาพไว้ในไฟล์ HTML ไฟล์เดียว `outputs/super_viz.html` โหลด plotly.js จาก CDN ครั้งเดียว เปิดในเบราว์เซอร์ได้เลย

In [13]:
PAGE_FIGURES = [
    ("A day on BART", "12_day_on_bart"),
    ("Where people travel", "12_od_flows"),
    ("12 years in one picture", "12_twelve_years"),
    ("Forecast explorer, daily", "12_forecast_explorer"),
    ("Forecast explorer, notable weeks", "12_event_weeks"),
    ("Station recovery race", "12_recovery_race"),
    ("Stations that changed role", "12_station_roles"),
    ("The Transbay Tube over 13 years", "12_transbay_years"),
]
figures_by_name = {
    "12_day_on_bart": day_figure,
    "12_od_flows": od_figure,
    "12_twelve_years": years_figure,
    "12_forecast_explorer": explorer_figure,
    "12_event_weeks": weeks_figure,
    "12_recovery_race": race_figure,
    "12_station_roles": roles_figure,
    "12_transbay_years": tube_figure,
}

sections = []
for position, (heading, name) in enumerate(PAGE_FIGURES):
    if position == 0:
        include = "cdn"
    else:
        include = False
    body = figures_by_name[name].to_html(full_html=False, include_plotlyjs=include, auto_play=False)
    sections.append(f"<section><h2>{heading}</h2>{body}</section>")

page = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>BART Ridership Visual Story</title>
<style>
  body {{ margin: 0; background: #ffffff; color: {viz.INK}; font-family: {viz.FONT}; }}
  header, section {{ max-width: 1500px; margin: 0 auto; padding: 16px 24px; }}
  header p {{ color: {viz.INK2}; max-width: 900px; line-height: 1.5; }}
  h1 {{ font-size: 30px; margin: 24px 0 8px; }}
  h2 {{ font-size: 15px; text-transform: uppercase; letter-spacing: 0.06em; color: {viz.MUTED}; margin: 32px 0 0;
        border-top: 1px solid {viz.GRID}; padding-top: 16px; }}
  section > div {{ overflow-x: auto; }}
</style>
</head>
<body>
<header>
<h1>BART ridership, 2011 - 2023</h1>
<p>Hourly origin-destination trips for all 50 stations: 112.5 million rows, 1.19 billion trips.
Every chart is interactive: hover for numbers, use the buttons and sliders, and press Play on the animated charts.
Forecasts come from the M1 model, made 7 days ahead in a quarter-by-quarter backtest.</p>
</header>
{"".join(sections)}
</body>
</html>"""

page_path = OUT / "super_viz.html"
page_path.write_text(page, encoding="utf-8")
print(f"wrote outputs/{page_path.name} ({page_path.stat().st_size / 1e6:.1f} MB)")

wrote outputs/super_viz.html (3.6 MB)


## สรุป

ภาพทั้ง 8 ชุดเล่าเรื่องเดียวกันจากหลายมุม
- โครงสร้างการเดินทางของ BART ยังเหมือนเดิม คือเช้าเข้าเมือง เย็นออกจากเมือง และ Tube ยังเป็นเส้นทางหลัก แต่ขนาดเหลือราว 1 ใน 3 ของปี 2019 (ภาพ 1, 2, 3, 8)
- การหายไปกระจุกที่ย่านทำงานและการเดินทางไกลเข้าเมือง ส่วนสนามบินและปลายสายบางแห่งฟื้นดีกว่า และสถานีครึ่งหนึ่งเปลี่ยนบทบาท (ภาพ 6, 7)
- โมเดลพยากรณ์ล่วงหน้า 7 วันแม่นในวันปกติ และวันที่พลาดคือวันที่มีงานหรือเหตุผิดปกติ ซึ่งดูรายละเอียดได้ในตัวสำรวจ (ภาพ 4, 5)

ไฟล์ที่ได้
- `outputs/super_viz.html` หน้ารวมทุกภาพ เปิดในเบราว์เซอร์ (ต้องต่อเน็ตเพื่อโหลด plotly.js และแผนที่)
- `outputs/figures/12_*.html` และ `12_*.png` ภาพแยก (PNG ของภาพเคลื่อนไหวเก็บแค่สถานะเริ่มต้น)